# 06 - Evaluation

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [ ]:
# Load cleaned data
df = pd.read_csv("cleaned.csv", dtype={"CloseMonth": str})
months = sorted(df["CloseMonth"].unique())
df.shape

## Features

In [ ]:
# Same features as 05_feature_engineering
df["BedBathRatio"] = (df["BedroomsTotal"] / df["BathroomsTotalInteger"]).replace(float("inf"), 0).fillna(0)
df["Age"] = df["CloseMonth"].str[:4].astype(int) - df["YearBuilt"]

districts = gpd.read_file("school_districts/DistrictAreas2425.shp")
points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]), index=df.index, crs="EPSG:4326")
points = points.to_crs(districts.crs)
joined = gpd.sjoin(points, districts[["DistrictNa", "DistrictTy", "geometry"]], how="left", predicate="within")

# Homes in both an elementary and a high school district: keep the unified district if there is one, otherwise the elementary one
joined["priority"] = joined["DistrictTy"].map({"Unified": 0, "Elementary": 1, "High": 2})
joined = joined.sort_values("priority").groupby(level=0).first()
df["SchoolDistrict"] = joined["DistrictNa"].fillna("None")

df = pd.get_dummies(df, columns=["SchoolDistrict"], drop_first=True, dtype=int)
features = [c for c in df.columns if c not in ["ClosePrice", "CloseMonth"]]
df.shape

## Train/test split

In [ ]:
# Test on the most recent month, train on the X months before it
X = 4
train = df[df["CloseMonth"].isin(months[-1 - X:-1])]
test = df[df["CloseMonth"] == months[-1]].copy()
train.shape, test.shape

In [ ]:
# Normalize numeric features using the training set's mean and std
num_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
            "GarageSpaces", "ParkingTotal", "AssociationFee", "Latitude", "Longitude", "BedBathRatio", "Age"]
train = train.copy()
mean, std = train[num_cols].mean(), train[num_cols].std()
train[num_cols] = (train[num_cols] - mean) / std
test[num_cols] = (test[num_cols] - mean) / std

## Models

In [ ]:
# XGBoost uses the best settings from 05_advanced_models
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(max_depth=6, learning_rate=0.1, n_estimators=600, random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(train[features], train["ClosePrice"])
    test[name] = model.predict(test[features])

## Metrics

In [ ]:
# Percent error for each home
def ape(actual, pred):
    return np.abs(pred - actual) / actual * 100

test["PriceBand"] = pd.cut(test["ClosePrice"], [0, 500_000, 1_000_000, 2_000_000, np.inf],
                           labels=["Under $500K", "$500K-$1M", "$1M-$2M", "Over $2M"])

rows = []
for name in models:
    errors = ape(test["ClosePrice"], test[name])
    rows.append((name, "All", len(test), r2_score(test["ClosePrice"], test[name]), errors.mean(), errors.median()))
    for band in test["PriceBand"].cat.categories:
        band_errors = errors[test["PriceBand"] == band]
        rows.append((name, band, len(band_errors), np.nan, band_errors.mean(), band_errors.median()))

summary = pd.DataFrame(rows, columns=["model", "price_band", "homes", "R2", "MAPE", "MdAPE"])

In [ ]:
for name in models:
    print(name)
    for _, r in summary[summary["model"] == name].iterrows():
        if r["price_band"] == "All":
            print(f"All homes ({r['homes']:,}): R2 {r['R2']:.3f}, MAPE {r['MAPE']:.1f}%, MdAPE {r['MdAPE']:.1f}%")
        else:
            print(f"{r['price_band']} ({r['homes']:,}): MAPE {r['MAPE']:.1f}%, MdAPE {r['MdAPE']:.1f}%")
    print()

In [ ]:
summary.round(3).to_csv("metrics_summary.csv", index=False)

## Insights

- **Best model depends on the metric.** XGBoost has the highest R² (0.846), but Random Forest has the lowest percent errors (MAPE 18.0%, MdAPE 9.0%). R² is driven by dollar errors, so it rewards getting expensive homes right. MAPE and MdAPE weight every home equally.
- **$500K-$1M is the easiest band.** It has the most homes (about 40% of the test set) and the lowest errors: Random Forest is within 6.9% for the typical home.
- **Under $500K has the biggest gap between MAPE and MdAPE.** For Random Forest the typical home is off by 9.0%, but the average is 44.1%. A small number of cheap homes are predicted far too high, and since the price is small, the percent error is huge.
- **Over $2M is consistently harder.** Errors rise to around 15% for the typical home. These homes are rarer and vary more in ways the features don't capture, like views or finishes.
- **Linear Regression trails in every band**, especially under $500K (MdAPE 48.2%), where a single straight-line fit can't handle cheap and expensive areas at the same time.
